# 音频零基础 5/6：叠加、谐波、噪声与 SNR

这节课从生活中的声音出发，再落到 Python 数组。**先建立直觉，再看公式，最后用代码验证。**

| 项目 | 内容 |
|---|---|
| 本课要回答的问题 | 复杂声音怎样由简单成分叠加，噪声强弱又怎样量化？ |
| 前置要求 | 会运行 Notebook；看不懂代码时先读 Python/PyTorch 基础 1～3 |
| 建议投入 | 90～120 分钟，分成“直觉”和“代码”两次完成也可以 |
| 四个关键词 | 叠加、谐波、噪声、SNR |
| 通关证据 | 能用自己的话解释、能算一个数字例子、能画图验证、能从空白实现核心函数 |

本课不是词汇表。每出现一个量，都要写清楚：**它描述什么、单位是什么、数组中怎样表示、改变后听觉或图形怎样变化。**


## 课前回忆：先写猜测，不查答案

1. 对问题“复杂声音怎样由简单成分叠加，噪声强弱又怎样量化？”写下你现在的答案；不会就写“不知道”。
2. 四个关键词中，圈出最陌生的一个：叠加 / 谐波 / 噪声 / SNR。
3. 画一条横轴是时间的线，标出 0 秒、0.5 秒和 1 秒。
4. 写下一个可验证的预测：如果把声音的某个参数加倍，图或听感会怎样？

学完后回到这里，用另一种颜色修正。保留错误猜测，它是学习证据。


## 固定观察框架：物理世界 → 数字 → 图 → 听感

```text
声源振动 → 空气压力随时间变化 → 麦克风电信号 → 离散采样值 x[n]
                                                ↓
                                      波形 / 数值统计 / 频谱
```

后面所有 ASR 前端课都沿这条链展开。波形不是声音本身，而是麦克风在一串离散时刻记录下来的数字。


## 1. 波形可以逐点相加

线性叠加是频谱、滤波、回声和多麦克风处理的共同起点。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sr = 16_000
t = np.arange(sr) / sr
fundamental = 0.6 * np.sin(2*np.pi*200*t)
second_harmonic = 0.2 * np.sin(2*np.pi*400*t)
third_harmonic = 0.1 * np.sin(2*np.pi*600*t)
mixture = fundamental + second_harmonic + third_harmonic

plt.plot(t[:400]*1000, fundamental[:400], label="200 Hz")
plt.plot(t[:400]*1000, mixture[:400], label="mixture", alpha=0.8)
plt.xlabel("time (ms)"); plt.ylabel("amplitude"); plt.legend(); plt.grid(True); plt.show()


## 2. 谐波是基频整数倍附近的成分

真实语音比这个例子复杂，但“基频 + 谐波 + 噪声”的模型能建立重要直觉。


In [ ]:
frequency_hz = np.fft.rfftfreq(len(mixture), d=1/sr)
magnitude = np.abs(np.fft.rfft(mixture))
top = np.argsort(magnitude)[-6:][::-1]
for index in top:
    print(f"{frequency_hz[index]:7.1f} Hz magnitude={magnitude[index]:.1f}")


## 3. SNR 比较信号功率与噪声功率

SNR 也必须说明信号与噪声怎样定义。正 dB 表示信号功率更强，0 dB 表示二者功率相等。


In [ ]:
def power(signal: np.ndarray) -> float:
    signal = np.asarray(signal, dtype=np.float64)
    if signal.size == 0:
        raise ValueError("signal must not be empty")
    return float(np.mean(signal ** 2))


def snr_db(clean: np.ndarray, noise: np.ndarray) -> float:
    noise_power = power(noise)
    if noise_power == 0:
        return float("inf")
    return float(10 * np.log10(power(clean) / noise_power))


rng = np.random.default_rng(7)
noise = rng.normal(size=len(mixture))
noise *= np.sqrt(power(mixture) / power(noise))
print("equal-power SNR:", snr_db(mixture, noise), "dB")


## 4. 按目标 SNR 缩放噪声

只改噪声比例，不改干净信号，才能进行受控实验。


In [ ]:
def mix_at_snr(clean: np.ndarray, raw_noise: np.ndarray, target_snr_db: float):
    if clean.shape != raw_noise.shape:
        raise ValueError("clean and noise must have the same shape")
    target_noise_power = power(clean) / (10 ** (target_snr_db / 10))
    scale = np.sqrt(target_noise_power / power(raw_noise))
    scaled_noise = raw_noise * scale
    return clean + scaled_noise, scaled_noise


for target in [20, 10, 0, -5]:
    noisy, scaled_noise = mix_at_snr(mixture, rng.normal(size=len(mixture)), target)
    print(f"target={target:>3} dB, measured={snr_db(mixture, scaled_noise):7.3f} dB, peak={np.max(np.abs(noisy)):.3f}")


## 分层练习：不要一次做完

### A. 直觉与单位（每题 1 分）

1. 不看上文，用一句话定义：叠加。
2. 为 `谐波` 写出单位；如果它没有单位，要明确说明。
3. 举一个生活中的例子解释 `噪声`。
4. 画图说明 `SNR` 增大时，横轴或纵轴怎样变化。

### B. 数字与预测（每题 2 分）

5. 自己构造一个包含具体数字的计算例子，并标出每一步单位。
6. 把本课一个参数改成 0.5 倍和 2 倍；运行前先画出预期图形。
7. 找出一个“代码能运行，但物理含义错误”的输入，解释为什么错误。
8. 用 `shape / dtype / min / max` 四项审计本课最重要的数组。

### C. 实现与迁移（每题 3 分）

9. 从空白实现：**实现 snr_db 与 mix_at_snr，并用测量值验证三个目标 SNR**，不能复制上面的函数。
10. 为实现写正常、边界和错误输入三类测试。
11. 换一组参数或换一条音频，验证结论是否仍成立。
12. 用 90 秒向没学过编程的人解释本课，只允许使用两个术语，并必须包含一个数字例子。

满分 24 分。达到 19 分且第 9～10 题完成，才建议继续；15～18 分次日重做；低于 15 分先回到图和单位。


## 最小掌握门禁

- [ ] 我能把本课每个量说成“含义 + 单位 + 数组表示”。
- [ ] 我能在运行前预测参数变化的方向。
- [ ] 我能从空白完成核心实现并覆盖边界输入。
- [ ] 我能说出一个常见误解，以及用什么证据推翻它。
- [ ] 我已把错题写入根目录 `LEARNING_LOG.md`，并安排明天、7 天、30 天复习。

下一步：音频基础 6：读取、试听、可视化和审计真实 WAV。
